# Algorithmic Model Trading Strategy

This notebook is the algorithmic counterpart to `manual_auction_strategy.ipynb`. It documents the Round 1 model for `ASH_COATED_OSMIUM` and `INTARIAN_PEPPER_ROOT`, explains why the edge should exist, records the anti-overfitting protocol, and links every conclusion back to reproducible backtests.


## Final Strategy

Upload `trader.py` for the algorithmic submission.

```text
INTARIAN_PEPPER_ROOT
- Fair value: live_intercept + 0.001 * timestamp
- Build toward +80 inventory early while asks are no more than 8 XIRECs above fair
- Begin patient exit after timestamp 995000 with exit edge 5
- Force a wider final exit after timestamp 998000 with exit edge 8
- Goal: capture the repeatable intraday drift while ending flat

ASH_COATED_OSMIUM
- Fair value: anchored EMA around 10000 plus a small order-book imbalance adjustment
- Cross only when edge is at least 4 XIRECs
- Passive quote conservatively around fair value
- Goal: secondary mean-reversion/spread capture, not required for the 200k target
```

The selected parameters were chosen using only days `-2` and `-1`. Day `0` is treated as a local holdout run because the live evaluation is expected to be day `1`.


In [1]:
import csv, json, math, statistics, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'trader.py').exists() and (ROOT.parent / 'trader.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data' / 'round1'
DIAGNOSTICS = json.loads((ROOT / 'logs' / 'round1_diagnostics.json').read_text())

price_rows = []
for path in sorted(DATA.glob('prices_round_1_day_*.csv')):
    with path.open(newline='') as f:
        for row in csv.DictReader(f, delimiter=';'):
            parsed = {
                'day': int(row['day']),
                'timestamp': int(row['timestamp']),
                'product': row['product'],
                'mid_price': float(row['mid_price']),
            }
            for level in (1, 2, 3):
                for side in ('bid', 'ask'):
                    parsed[f'{side}_price_{level}'] = float(row[f'{side}_price_{level}']) if row[f'{side}_price_{level}'] else None
                    parsed[f'{side}_volume_{level}'] = float(row[f'{side}_volume_{level}']) if row[f'{side}_volume_{level}'] else None
            price_rows.append(parsed)
products = sorted({row['product'] for row in price_rows})
print(f'Loaded {len(price_rows)} price rows')
print('Products:', ', '.join(products))


Loaded 60000 price rows
Products: ASH_COATED_OSMIUM, INTARIAN_PEPPER_ROOT


## Data Quality and Product Shape

The dataset has three historical days: `-2`, `-1`, and `0`. For model selection we use `-2` and `-1`; day `0` is reserved as the holdout test run.

The products behave differently enough that a single generic model would be a bad fit. Pepper root has a large deterministic-looking drift. Osmium is much tighter and mostly stationary around 10,000.


In [2]:
for product in products:
    print(product)
    for day in sorted({row['day'] for row in price_rows}):
        mids = [row['mid_price'] for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] > 0]
        zeros = sum(1 for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] <= 0)
        print(
            f' day {day:2d}: n={len(mids)} zeros={zeros} mean={statistics.mean(mids):.2f} '
            f'std={statistics.pstdev(mids):.2f} min={min(mids):.1f} max={max(mids):.1f} '
            f'first={mids[0]:.1f} last={mids[-1]:.1f}'
        )
    print()


ASH_COATED_OSMIUM
 day -2: n=9982 zeros=18 mean=9998.17 std=5.22 min=9979.0 max=10019.0 first=10010.0 last=9993.5
 day -1: n=9983 zeros=17 mean=10000.83 std=4.45 min=9982.0 max=10019.0 first=10003.0 last=10002.0
 day  0: n=9986 zeros=14 mean=10001.61 std=5.68 min=9977.0 max=10023.0 first=10013.0 last=10007.0

INTARIAN_PEPPER_ROOT
 day -2: n=9984 zeros=16 mean=10499.96 std=288.71 min=9998.5 max=11003.0 first=9998.5 last=11001.5
 day -1: n=9983 zeros=17 mean=11500.03 std=288.65 min=10995.0 max=12006.0 first=10998.5 last=11998.0
 day  0: n=9979 zeros=21 mean=12500.17 std=288.72 min=11994.0 max=13007.0 first=11998.5 last=13000.0



## Model Findings

**Finding 1: Pepper root follows an affine fair-value path.** Same-timestamp prices are almost perfectly aligned across days, and each day is roughly 1,000 XIRECs higher than the prior day. The intraday slope is approximately `0.001 * timestamp`.

**Finding 2: Osmium is not a trend product in this sample.** It clusters tightly around 10,000. Its short-term signal is microstructure: deviations from a local EMA mean-revert, while order-book imbalance helps predict the next tick direction.

These findings are intentionally simple. The round has only three days of history, so a flexible ML model would have more degrees of freedom than evidence. The model uses structural relationships that can be stated, tested, and stress checked.


In [3]:
def corr(xs, ys):
    mx, my = statistics.mean(xs), statistics.mean(ys)
    sx = sum((x - mx) ** 2 for x in xs)
    sy = sum((y - my) ** 2 for y in ys)
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / (sx * sy) ** 0.5 if sx and sy else 0.0

for product in products:
    print(product)
    by_day = {}
    for day in (-2, -1, 0):
        by_day[day] = {row['timestamp']: row['mid_price'] for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] > 0}
    for left, right in [(-2, -1), (-1, 0)]:
        stamps = sorted(set(by_day[left]).intersection(by_day[right]))
        xs = [by_day[left][stamp] for stamp in stamps]
        ys = [by_day[right][stamp] for stamp in stamps]
        print(f' day-pair {left}/{right}: same-timestamp corr={corr(xs, ys):.4f} mean_diff={statistics.mean(y - x for x, y in zip(xs, ys)):.2f}')

    ema_dev, imbalance, next_delta = [], [], []
    for day in (-2, -1, 0):
        rows = [row for row in price_rows if row['product'] == product and row['day'] == day and row['mid_price'] > 0]
        rows.sort(key=lambda row: row['timestamp'])
        ema = rows[0]['mid_price']
        for i, row in enumerate(rows[:-1]):
            bid, ask = row['bid_price_1'], row['ask_price_1']
            bid_volume, ask_volume = row['bid_volume_1'], row['ask_volume_1']
            if bid is None or ask is None:
                continue
            ema_dev.append(row['mid_price'] - ema)
            imbalance.append((bid_volume - ask_volume) / (bid_volume + ask_volume))
            next_delta.append(rows[i + 1]['mid_price'] - row['mid_price'])
            ema = 0.8 * ema + 0.2 * row['mid_price']
    print(f' signal correlations: ema_deviation_vs_next_delta={corr(ema_dev, next_delta):.4f} imbalance_vs_next_delta={corr(imbalance, next_delta):.4f}')
    print()


ASH_COATED_OSMIUM
 day-pair -2/-1: same-timestamp corr=-0.0598 mean_diff=2.66
 day-pair -1/0: same-timestamp corr=-0.0644 mean_diff=0.78
 signal correlations: ema_deviation_vs_next_delta=-0.4077 imbalance_vs_next_delta=0.3809

INTARIAN_PEPPER_ROOT
 day-pair -2/-1: same-timestamp corr=0.9999 mean_diff=1000.01
 day-pair -1/0: same-timestamp corr=0.9999 mean_diff=999.99
 signal correlations: ema_deviation_vs_next_delta=-0.4530 imbalance_vs_next_delta=0.3849



## Trading Theorems

**Pepper drift theorem.** If a product follows `price(t) = intercept + slope * t + noise`, and `slope > 0`, then the expected value of inventory held from early day to late day is positive. With an 80-unit limit and a slope of about 1,000 XIRECs per day, the gross ceiling is close to 80,000 XIRECs per day before spread, fill, and exit costs.

**Flattening theorem.** A mark-to-market backtest can overstate robustness if it finishes with large inventory. The staged exit sacrifices some paper PnL to end flat on all observed days, which makes the result less dependent on terminal marks and more likely to transfer.

**Osmium stationarity theorem.** If a product is centered around a stable anchor and local deviations have negative correlation with next price changes, then conservative mean reversion is justified. Because the edge is small, the model only crosses with a wide threshold and treats passive quotes as optional upside.


## Overfitting Controls

The model deliberately avoids linear regression, k-nearest neighbors, neural networks, or high-dimensional features. The anti-overfit process is:

1. Use days `-2` and `-1` for parameter selection.
2. Reserve day `0` as the local holdout test run.
3. Choose a robust score, not raw all-days max PnL.
4. Penalize parameter choices away from the center of the observed plateau.
5. Require flat end-of-day inventory.
6. Test a neighborhood grid around the chosen parameters.
7. Apply Monte Carlo execution stress.

The selected parameter set ranks `1 / 48` by train-only robust score but only `7 / 48` by all-days combined PnL. That is good: it means day `0` was not used to choose the historical maximum.


In [4]:
grid = DIAGNOSTICS['parameter_grid']
mc = DIAGNOSTICS['monte_carlo']
selected = grid['selected']
print('selection protocol:', grid['selection_protocol'])
print('deterministic combined pnl:', DIAGNOSTICS['deterministic_backtest']['combined_pnl'])
print('train target pass rate:', grid['train_target_pass_rate'])
print('all-days target pass rate:', grid['target_pass_rate'])
print('selected train rank:', grid['selected_train_rank'], 'of', grid['summary']['count'])
print('selected all-days combined rank:', grid['selected_combined_rank'], 'of', grid['summary']['count'])
print('selected train total:', selected['train_total_pnl'])
print('selected holdout day 0:', selected['holdout_day0_pnl'])
print('selected end positions:', selected['end_positions'])
print('selected neighborhood train summary:', grid['selected_neighborhood_train_summary'])
print('mc summary:', mc['summary'])
print('mc probability >= 200k:', mc['probability_above_200k'])
print('Gelman-Rubin R-hat:', mc['gelman_rubin_rhat'])
print('Geweke:', mc['geweke'])
print('Anderson-Darling:', mc['anderson_darling_normality'])
print('K-S:', mc['kolmogorov_smirnov_normality'])


selection protocol: Select parameters using only days -2 and -1. Day 0 is treated as a holdout test run, because live submission is expected to be evaluated on day 1.
deterministic combined pnl: 236444.0
train target pass rate: 1.0
all-days target pass rate: 1.0
selected train rank: 1 of 48
selected all-days combined rank: 7 of 48
selected train total: 157670.0
selected holdout day 0: 78774.0
selected end positions: {'-2': {'ASH_COATED_OSMIUM': 0, 'INTARIAN_PEPPER_ROOT': 0}, '-1': {'ASH_COATED_OSMIUM': 0, 'INTARIAN_PEPPER_ROOT': 0}, '0': {'ASH_COATED_OSMIUM': 0, 'INTARIAN_PEPPER_ROOT': 0}}
selected neighborhood train summary: {'count': 18, 'min': 157353.0, 'p05': 157353.0, 'p10': 157447.5, 'median': 157594.0, 'mean': 157599.66666666666, 'p90': 157776.0, 'p95': 157776.0, 'max': 157776.0, 'stdev': 130.72022711798576}
mc summary: {'count': 160, 'min': 236125.0, 'p05': 236152.94999999998, 'p10': 236163.0, 'median': 236209.0, 'mean': 236207.25625, 'p90': 236246.0, 'p95': 236256.2, 'max': 23

## Stock-Style Backtest Metrics

The diagnostics report raw XIREC PnL, while Sharpe and Sortino need a return series. The table below uses a synthetic capital base of `80 * day-start mid` for each traded product, summed across products. That gives a consistent stock-style denominator without pretending this is a real brokerage account.

The `252`-day scaling is included only as a familiar market convention. With three competition days, the daily PnL table, flat closing inventory, holdout split, parameter grid, and Monte Carlo stress are more important than the absolute annualized Sharpe value.


In [5]:
def pct(value):
    return 'n/a' if value is None or (isinstance(value, float) and math.isnan(value)) else f'{100 * value:,.2f}%'


def num(value):
    if isinstance(value, str):
        return value
    if value is None:
        return 'n/a'
    if isinstance(value, float) and math.isnan(value):
        return 'n/a'
    if isinstance(value, float) and math.isinf(value):
        return 'inf'
    return f'{value:,.2f}'


def drawdown_from_levels(levels):
    peak = levels[0]
    max_drawdown = 0.0
    for level in levels:
        peak = max(peak, level)
        if peak:
            max_drawdown = min(max_drawdown, level / peak - 1)
    return max_drawdown


def downside_deviation(returns, minimum_acceptable_return=0.0):
    downside = [min(0.0, value - minimum_acceptable_return) for value in returns]
    return math.sqrt(sum(value * value for value in downside) / len(returns)) if returns else float('nan')


def print_table(headers, rows):
    widths = [len(header) for header in headers]
    for row in rows:
        widths = [max(width, len(str(value))) for width, value in zip(widths, row)]
    print(' | '.join(header.ljust(width) for header, width in zip(headers, widths)))
    print(' | '.join('-' * width for width in widths))
    for row in rows:
        print(' | '.join(str(value).ljust(width) for value, width in zip(row, widths)))


position_limits = {'ASH_COATED_OSMIUM': 80, 'INTARIAN_PEPPER_ROOT': 80}
day_product_rows = {}
for row in price_rows:
    if row['mid_price'] <= 0:
        continue
    day_product_rows.setdefault((row['day'], row['product']), []).append(row)
for rows in day_product_rows.values():
    rows.sort(key=lambda row: row['timestamp'])

day_results = DIAGNOSTICS['deterministic_backtest']['day_results']
strategy_rows = []
daily_pnls = []
daily_returns = []
for result in day_results:
    day = result['day']
    capital_base = sum(position_limits[product] * day_product_rows[(day, product)][0]['mid_price'] for product in products)
    pnl = result['total_pnl']
    daily_pnls.append(pnl)
    daily_return = pnl / capital_base
    daily_returns.append(daily_return)
    strategy_rows.append([
        day,
        num(capital_base),
        num(pnl),
        pct(daily_return),
        num(result['pnl_by_product']['INTARIAN_PEPPER_ROOT']),
        num(result['pnl_by_product']['ASH_COATED_OSMIUM']),
        str(result['position']),
    ])

print('Strategy daily backtest data')
print_table(['day', 'capital_base', 'pnl', 'return', 'pepper_pnl', 'osmium_pnl', 'end_position'], strategy_rows)

mean_return = statistics.mean(daily_returns)
return_volatility = statistics.pstdev(daily_returns) if len(daily_returns) > 1 else 0.0
mean_pnl = statistics.mean(daily_pnls)
pnl_volatility = statistics.pstdev(daily_pnls) if len(daily_pnls) > 1 else 0.0
downside = downside_deviation(daily_returns)
annual_factor = math.sqrt(252)
sharpe = mean_return / return_volatility * annual_factor if return_volatility else float('inf')
sortino = mean_return / downside * annual_factor if downside else float('inf')

equity_curve = []
running_pnl = 0.0
for pnl in daily_pnls:
    running_pnl += pnl
    equity_curve.append(running_pnl)
max_strategy_drawdown = drawdown_from_levels([0.0] + equity_curve)
calmar = (mean_return * 252) / abs(max_strategy_drawdown) if max_strategy_drawdown else float('inf')

metric_rows = [
    ['combined_pnl', num(DIAGNOSTICS['deterministic_backtest']['combined_pnl'])],
    ['mean_daily_pnl', num(mean_pnl)],
    ['daily_pnl_stdev', num(pnl_volatility)],
    ['positive_day_rate', pct(sum(pnl > 0 for pnl in daily_pnls) / len(daily_pnls))],
    ['target_day_hit_rate', pct(sum(pnl >= 200_000 / 3 for pnl in daily_pnls) / len(daily_pnls))],
    ['filled_orders', DIAGNOSTICS['deterministic_backtest']['filled_orders']],
    ['filled_quantity', DIAGNOSTICS['deterministic_backtest']['filled_quantity']],
    ['pnl_per_filled_unit', num(DIAGNOSTICS['deterministic_backtest']['combined_pnl'] / DIAGNOSTICS['deterministic_backtest']['filled_quantity'])],
    ['mean_daily_return_on_capital_base', pct(mean_return)],
    ['daily_return_volatility', pct(return_volatility)],
    ['annualized_sharpe_252_day_convention', num(sharpe)],
    ['annualized_sortino_252_day_convention', 'inf (no negative-return days)' if math.isinf(sortino) else num(sortino)],
    ['cumulative_pnl_max_drawdown', pct(max_strategy_drawdown)],
    ['calmar_252_day_convention', 'inf (no drawdown in cumulative PnL)' if math.isinf(calmar) else num(calmar)],
]

print()
print('Strategy summary metrics')
print_table(['metric', 'value'], metric_rows)

price_metric_rows = []
for product in products:
    for day in sorted({row['day'] for row in price_rows}):
        rows = day_product_rows[(day, product)]
        mids = [row['mid_price'] for row in rows]
        returns = [mids[i] / mids[i - 1] - 1 for i in range(1, len(mids)) if mids[i - 1] > 0]
        realized_volatility = statistics.pstdev(returns) * math.sqrt(len(returns)) if len(returns) > 1 else 0.0
        day_return = mids[-1] / mids[0] - 1
        max_drawdown = drawdown_from_levels(mids)
        trend_to_volatility = day_return / realized_volatility if realized_volatility else float('inf')
        price_metric_rows.append([
            product,
            day,
            num(mids[0]),
            num(mids[-1]),
            pct(day_return),
            pct(realized_volatility),
            pct(max_drawdown),
            num(trend_to_volatility),
        ])

print()
print('Stock-like mid-price metrics')
print_table(['product', 'day', 'start_mid', 'end_mid', 'day_return', 'realized_vol', 'max_drawdown', 'trend_to_vol'], price_metric_rows)


Strategy daily backtest data
day | capital_base | pnl       | return | pepper_pnl | osmium_pnl | end_position                                       
--- | ------------ | --------- | ------ | ---------- | ---------- | ---------------------------------------------------
-2  | 1,600,680.00 | 78,813.00 | 4.92%  | 78,813.00  | 0.00       | {'ASH_COATED_OSMIUM': 0, 'INTARIAN_PEPPER_ROOT': 0}
-1  | 1,680,120.00 | 78,857.00 | 4.69%  | 78,857.00  | 0.00       | {'ASH_COATED_OSMIUM': 0, 'INTARIAN_PEPPER_ROOT': 0}
0   | 1,760,920.00 | 78,774.00 | 4.47%  | 78,774.00  | 0.00       | {'ASH_COATED_OSMIUM': 0, 'INTARIAN_PEPPER_ROOT': 0}

Strategy summary metrics
metric                                | value                              
------------------------------------- | -----------------------------------
combined_pnl                          | 236,444.00                         
mean_daily_pnl                        | 78,814.67                          
daily_pnl_stdev                       | 3

## Why This Should Be Profitable

The 200,000 XIREC target over three days is approximately 66,667 XIRECs per day. The pepper-root drift strategy alone backtests near 78,800 XIRECs per day with flat closing inventory. That leaves a margin of roughly 12,000 XIRECs per day before considering any osmium contribution.

The most important risk is that live day `1` breaks the pepper-root affine drift. The model mitigates that by recalibrating the intercept from the live book rather than hard-coding a day number. If the slope persists, the strategy should transfer. If the slope disappears, the staged exit limits the time spent holding late-day inventory.


## Reproducibility

Run the algorithmic diagnostics from the repo root:

```powershell
python scripts\round1_diagnostics.py
```

The script writes `logs/round1_diagnostics.json`, which this notebook reads for final backtest, parameter-grid, Monte Carlo, Geweke, Gelman-Rubin, Anderson-Darling, and K-S results.
